# From Transformer Block to Mini-GPT

> In the previous section, we assembled Attention, FFN, residual connections, and LayerNorm into a Transformer Block. The Block receives input of shape `[batch, seq, d_model]` and outputs the same shape — without changing dimensions, it lets tokens exchange information and process each one individually.
>
> A single Block performs one round of transformation. GPT stacks many such layers (GPT-2 has 12 layers, GPT-3 has 96), and each layer further integrates context on top of the previous one. In this section, we embed the Transformer Block into a complete model: adding Embedding and position encoding at the front, and an output projection at the end, to build a Mini-GPT.

At the end of the previous section, we had a set of independent components: the Embedding layer turns token IDs into vectors, Position Encoding adds position information, and the Transformer Block lets tokens exchange information. These components each work on their own, but they haven't been connected into a complete pipeline.

What GPT does can be summarized in one sentence: given a prefix, predict the next token. Data flows through the entire pipeline:

```
Token IDs → Embedding + Position → N layers of Block → LayerNorm → Linear → logits
```

The intermediate dimensions stay at `d_model` throughout; only the final step projects to the vocabulary size. In terms of shapes: `[batch, seq]` → `[batch, seq, d_model]` → `...` → `[batch, seq, vocab_size]`.

Let's start with GPT-2's public configuration to see where all the parameters go in this pipeline.

In [ ]:
# Using GPT-2 Small's publicly available configuration directly
gpt2_config = {
    "vocab_size": 50257,
    "n_positions": 1024,
    "n_embd": 768,
    "n_layer": 12,
    "n_head": 12,
}

vocab_size = gpt2_config["vocab_size"]
n_positions = gpt2_config["n_positions"]
n_embd = gpt2_config["n_embd"]
n_layer = gpt2_config["n_layer"]
n_head = gpt2_config["n_head"]
d_ff = 4 * n_embd

print("=== GPT-2 Small Configuration (public structural params) ===")
print(f"Vocabulary size:    {vocab_size}")
print(f"Hidden dimension:   {n_embd}")
print(f"Number of layers:   {n_layer}")
print(f"Attention heads:    {n_head}")
print(f"Max sequence length:{n_positions}")

GPT-2's parameters are mainly distributed across three parts: the Embedding tables, each Transformer Block layer, and the final LayerNorm. Let's first calculate the Embedding part.

In [ ]:
# Embedding layer parameters: Token Embedding + Position Embedding
wte_params = vocab_size * n_embd   # Token Embedding table: one d_model-dim vector per token
wpe_params = n_positions * n_embd  # Position Embedding table: one d_model-dim vector per position

print(f"Token Embedding (wte):    {wte_params:>10,}")
print(f"Position Embedding (wpe): {wpe_params:>10,}")
print(f"Embedding total:          {wte_params + wpe_params:>10,}")
print()
print("Key observation: Embedding accounts for nearly 40% of GPT-2's parameters, but GPT-2 shares weights between wte and lm_head, so it doesn't add extra cost.")

Next, let's calculate the parameters in each Transformer Block. A Block contains two LayerNorm layers, one Attention module, and one MLP.

In [ ]:
# LayerNorm parameters: weight + bias, each d_model in size
ln_params = 2 * n_embd

# GPT-2's attention: c_attn computes Q/K/V in one shot, then c_proj
attn_params = n_embd * (3 * n_embd) + (3 * n_embd)   # c_attn: combined Q/K/V projection
attn_params += n_embd * n_embd + n_embd               # c_proj: output projection

# GPT-2's MLP: expand to 4x dimension, then compress back to n_embd
mlp_params = n_embd * d_ff + d_ff      # fc1: d_model → 4*d_model
mlp_params += d_ff * n_embd + n_embd   # fc2: 4*d_model → d_model

layer_total = ln_params + attn_params + ln_params + mlp_params

print("=== Parameters per Transformer Block ===")
print(f"LayerNorm 1:  {ln_params:>10,}")
print(f"Attention:    {attn_params:>10,}")
print(f"LayerNorm 2:  {ln_params:>10,}")
print(f"MLP:          {mlp_params:>10,}")
print(f"Per layer total: {layer_total:>10,}")
print()
print("Key observation: MLP accounts for about 2/3 of each layer's parameters, Attention about 1/3.")

Finally, add up all parts to get the total parameter count of GPT-2 Small.

In [ ]:
# Summary: Embedding + N layers of Block + final LayerNorm
ln_f_params = 2 * n_embd  # Final LayerNorm
total_unique_params = wte_params + wpe_params + n_layer * layer_total + ln_f_params

print("=== GPT-2 Small Parameter Summary ===")
print(f"Token Embedding (wte):       {wte_params:>10,}")
print(f"Position Embedding (wpe):    {wpe_params:>10,}")
print(f"{n_layer} layers of Block total:       {n_layer * layer_total:>10,}")
print(f"Final LayerNorm:             {ln_f_params:>10,}")
print(f"LM Head extra params:        {0:>10,}  ← GPT-2 shares weights with wte")
print(f"{'-' * 55}")
print(f"Total params (no double-counting shared weights): {total_unique_params:>10,}")
print()
print("Key observation: GPT-2 and MiniGPT share the same skeleton, but GPT-2 also uses learnable position encoding and weight sharing.")

Let's first lay out the differences between GPT-2 and the original Transformer. GPT-2 only keeps the Decoder backbone, used to continuously predict the next token; it has no Encoder and no Cross-Attention.

```text
GPT-2 / Decoder-Only

Token IDs
  ↓
Token Embedding + Position Embedding
  ↓
Masked Self-Attention   ← can only see current and earlier tokens
  ↓
Feed-Forward Network
  ↓
Repeat many layers of Decoder Block
  ↓
LM Head
  ↓
Predict next token
```

For comparison, the original Transformer uses an Encoder-Decoder architecture, commonly used for tasks like translation where you "read the input first, then generate the output":

```text
Original Transformer / Encoder-Decoder

Input sentence → Encoder → Encoder output
                         ↓
Target prefix → Decoder → Cross-Attention reads Encoder output
                         ↓
                      Generate next token
```

Comparing the two diagrams: the original Transformer's Decoder has two types of Attention — Masked Self-Attention (looking at its own prefix) and Cross-Attention (reading the Encoder's output). GPT-2 only has Masked Self-Attention, because it is a pure generative model with no Encoder.

In [ ]:
import torch

import numpy as np

torch.manual_seed(42)
np.random.seed(42)

## 1. Overall Structure of GPT

GPT's data flow can be viewed as a pipeline. The input is a token ID sequence, and the output is a prediction score for each position across the entire vocabulary:

```text
Token IDs
  ↓
Token Embedding + Position Embedding
  ↓
Transformer Block × N
  ↓
LayerNorm
  ↓
Linear (project to vocabulary size)
  ↓
logits: each position predicts the next token
```

In this pipeline, the intermediate dimensions stay at `d_model` throughout; only the final step expands to `vocab_size`. In terms of shapes:

```text
[batch, seq] → [batch, seq, d_model] → ... → [batch, seq, vocab_size]
```

Next, we'll implement this pipeline step by step.

## 2. Reusing the Transformer Block

To make this Notebook self-contained, we first bring in the three components from the previous section.

We won't re-explain the principles of each line here — just remember their responsibilities:

1. `MultiHeadAttention`: lets tokens see the context at the current position and earlier.
2. `FeedForward`: each token passes through a small network independently.
3. `TransformerBlock`: chains together Attention, FFN, Residual, and LayerNorm.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math
class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention; with a causal mask it becomes causal self-attention.

    Args:
        d_model: input/output dimension
        num_heads: number of attention heads
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head

        # Linear transforms for Q, K, V (merge num_heads into matrix operations)
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)

        # Output projection
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        """
        Input: x shape = [batch, seq_len, d_model]
        Output:   shape = [batch, seq_len, d_model]
        """
        batch_size, seq_len, _ = x.shape

        # 1. Linear transform + split into multi-head
        #    [batch, seq_len, d_model] → [batch, num_heads, seq_len, d_k]
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # 2. Attention scores: Q @ K^T
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 3. Mask (e.g., set future positions to -inf)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # 4. Softmax
        weights = F.softmax(scores, dim=-1)

        # 5. Weighted sum
        attn_output = weights @ V  # [batch, num_heads, seq_len, d_k]

        # 6. Concatenate heads back and project
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.d_model)
        return self.W_O(attn_output)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
class FeedForward(nn.Module):
    """FFN: two fully-connected layers, expand 4x then compress back"""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

class TransformerBlock(nn.Module):
    """A Transformer decoder layer: Attention + FFN, each with residual + LayerNorm"""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.attention(x, mask))  # Attention + Residual + Norm
        x = self.norm2(x + self.ffn(x))              # FFN + Residual + Norm
        return x

## 3. Implementing MiniGPT

In the previous section we confirmed that the Transformer Block receives input of shape `[batch, seq, d_model]` and outputs the same shape — it performs a "same-dimension transformation", integrating context information without changing the shape. Now we need to attach the input and output ends to form a complete model.

**Step 1: Token Embedding.** The input sequence is a set of integer token IDs, such as `[5, 12, 3]`. The integers themselves carry no semantic information (token ID 12 is not "greater than" token ID 3), so we need an Embedding table to map each ID to a `d_model`-dimensional vector. This was already implemented in notebook 03.

**Step 2: Position Encoding.** The same token appearing at position 1 versus position 10 may carry different meanings. Self-Attention has no inherent position information (it only looks at relationships between tokens), so we need to add position encoding after Embedding to let the model know "this vector comes from the Nth position in the sequence." Here we reuse the sinusoidal position encoding from the previous section (notebook 04).

**Step 3: Multi-layer Transformer Blocks.** After Embedding plus Position, the data passes through N Transformer Blocks sequentially. Each Block does the same thing: let tokens see each other, then process each one individually. The effect of stacking multiple layers is progressively richer information fusion — the first layer might learn relationships between adjacent words, while deeper layers may capture more distant, more abstract dependencies.

**Final Step: Output Projection.** After N layers of Blocks, each position yields a `d_model`-dimensional vector. A final linear layer projects it to the vocabulary size `vocab_size`, producing `logits` — the model's score for every token in the vocabulary at each position.

In [ ]:
# Reuse sinusoidal position encoding from the previous section
import torch
import torch.nn as nn
import math
def get_sinusoidal_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class MiniGPT(nn.Module):
    """Mini GPT: Embedding → N×TransformerBlock → LayerNorm → project to vocabulary"""
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=128):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = get_sinusoidal_encoding(max_seq_len, d_model)
        self.register_buffer('pe', pe)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)  # Project to vocabulary → logits

    def forward(self, x):
        # x: [batch, seq_len]  token IDs
        batch_size, seq_len = x.shape
        x = self.token_emb(x) + self.pe[:seq_len, :]          # Embedding + Position
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        mask = mask.view(1, 1, seq_len, seq_len)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_final(x)
        return self.lm_head(x)  # [batch, seq_len, vocab_size]

## 4. Forward Pass Data Flow

The model is defined. Now let's run a forward pass with random data.

The model hasn't been trained yet, so the output logits have no real meaning. This step only focuses on whether the data flows through the expected structure: input shape `[batch, seq]`, intermediate `[batch, seq, d_model]`, and final output `[batch, seq, vocab_size]`.

In [ ]:
# Test MiniGPT
import torch
vocab_size = 30
model = MiniGPT(vocab_size=vocab_size, d_model=64, num_heads=4, num_layers=2)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}, trainable: {trainable_params:,}")

# Forward pass: batch=2, seq_len=8
batch_input = torch.randint(0, vocab_size, (2, 8))
logits = model(batch_input)
print(f"\nInput: {batch_input.shape}  →  Output: {logits.shape}")
print(f"Output = [batch=2, seq=8, vocab={vocab_size}] → logits for each token at each position")
print(f"logits[0, 7, :5]: {logits[0, 7, :5].tolist()}  ← sample 0, position 7, scores for predicting next token")

In [ ]:
# Follow the architecture diagram: print shape at each step
import torch
vocab_size = 30
model = MiniGPT(vocab_size=vocab_size, d_model=32, num_heads=4, num_layers=2, max_seq_len=16)
idx = torch.randint(0, vocab_size, (2, 6))

with torch.no_grad():
    token_vec = model.token_emb(idx)
    pos_vec = model.pe[:idx.shape[1], :]
    x = token_vec + pos_vec
    print(f"1. token + position: {x.shape}")

    mask = torch.tril(torch.ones(idx.shape[1], idx.shape[1]))
    mask = mask.view(1, 1, idx.shape[1], idx.shape[1])

    for block_id, block in enumerate(model.blocks, start=1):
        x = block(x, mask)
        print(f"2.{block_id} After TransformerBlock: {x.shape}")

    x = model.ln_final(x)
    logits = model.lm_head(x)
    print(f"3. logits: {logits.shape}")

print("Key observation: intermediate dims stay at d_model throughout; only the final step changes to vocab_size.")

### Seeing the Model Structure with PyTorch

When writing PyTorch models in practice, the first thing to do is not to rush into training, but to confirm the shapes and parameter counts of each component.

Here we don't use any extra structure summary tools — just PyTorch's built-in `named_children()` and a single forward pass to print MiniGPT's main components. Focus on three things: how Embedding turns IDs into vectors, how TransformerBlock maintains `d_model`, and how `lm_head` finally projects to vocabulary size.

In [ ]:
import torch
def count_params(module):
    """Count the number of parameters in a PyTorch module"""
    return sum(p.numel() for p in module.parameters())

summary_model = MiniGPT(vocab_size=30, d_model=32, num_heads=4, num_layers=2, max_seq_len=16)
summary_input = torch.randint(0, 30, (2, 6))

print("=== MiniGPT Main Components ===")
for name, module in summary_model.named_children():
    params = count_params(module)
    print(f"{name:10s} | {module.__class__.__name__:16s} | params={params:,}")

with torch.no_grad():
    token_vec = summary_model.token_emb(summary_input)
    logits = summary_model(summary_input)

print()
print(f"Input token IDs:       {tuple(summary_input.shape)}")
print(f"Embedding output:      {tuple(token_vec.shape)}")
print(f"MiniGPT final logits:  {tuple(logits.shape)}")
print()
print("Key observation: Embedding turns [batch, seq] into [batch, seq, d_model].")
print("Intermediate TransformerBlocks keep d_model unchanged; lm_head finally projects to vocab_size.")

## 5. Logits and Predictions

The code in the previous section output a tensor called logits with shape `[batch, seq, vocab_size]`. This section explains what it is.

Logits are the raw scores output by the model. If the vocabulary has 30 tokens, then each position outputs 30 scores:

```text
logits[batch, position, token_id]
```

The higher the score, the more the model thinks this token is suitable as the "next token." For example, if `logits[0, 7, 12] = 2.3` is the highest score at position 7, it means the model (with current parameters) is inclined to output token ID 12 after position 7.

During training, Cross-Entropy Loss is used to increase the score of the correct answer and decrease the scores of wrong answers. During generation, softmax is applied to logits to get a probability distribution, then we sample from it or take the maximum. These topics will be expanded in later sections.

## 6. Karpathy's nanoGPT

The MiniGPT above is a simplified implementation for teaching. This section compares it against a real engineering codebase: Andrej Karpathy's open-source `nanoGPT`.

nanoGPT is a GPT-2-style PyTorch implementation with a small codebase that's easy to read. Its model architecture follows the GPT-2-style decoder-only Transformer design, and the repository also supports loading GPT-2 pretrained weights. For this section, nanoGPT's value is that it shares the same skeleton as our hand-written MiniGPT, but is written in a more engineering-oriented way. Comparing the two helps understand the difference between "teaching code" and "engineering code."

Below we run nanoGPT's tiny configuration with random initialization. It is not a GPT-2 pretrained model; it's only used to observe structure, shapes, loss, and parameter counts.

### Reading Order for `model.py`

When reading nanoGPT's `model.py`, the recommended order is:

```text
CausalSelfAttention   # self-attention with causal mask
MLP                   # small network each token passes through independently
Block                 # Attention + MLP + Residual + LayerNorm
GPT                   # embedding, blocks, ln_f, lm_head assembly
GPT.forward           # data flow of the entire model
```

These names map directly to our hand-written versions:

| Hand-written | nanoGPT Code | Purpose |
|:---|:---|:---|
| `MultiHeadAttention` | `CausalSelfAttention` | Multi-head causal self-attention |
| `FeedForward` | `MLP` | Per-token nonlinear processing |
| `TransformerBlock` | `Block` | One Transformer decoder block |
| `MiniGPT` | `GPT` | Complete decoder-only model |
| `token_emb` | `transformer.wte` | Token embedding table |
| `pe` / position | `transformer.wpe` | Position embedding table |
| `lm_head` | `lm_head` | Hidden state to vocab logits |

It's recommended to read `model.py` first to understand the model structure, then read the training script. Before the model structure is clear, the optimizer, checkpoint, and logging in the training loop can be distracting.

### The Main Thread of `GPT.forward`

`GPT.forward` is the most worthwhile function to read first in nanoGPT. It compresses the entire GPT data flow into a single line:

```text
idx
  → wte(idx)                 # token embedding
  → wpe(pos)                 # position embedding
  → drop(tok_emb + pos_emb)
  → for block in h           # multiple Transformer Blocks
  → ln_f
  → lm_head
  → logits / loss
```

This line shares the same skeleton as our hand-written MiniGPT. The engineering code adds a few implementation details:

1. nanoGPT uses a single linear layer `c_attn` to compute Q/K/V simultaneously, while our hand-written version uses three separate linear layers.
2. GPT-2 uses learnable `wpe`, while our hand-written version uses sinusoidal position encoding.
3. nanoGPT does weight tying: `wte.weight` and `lm_head.weight` share the same parameters.
4. Optimizer, checkpoint, and sampling are engineering details for training and inference; they don't change the main model thread.

### Running the nanoGPT Tiny Configuration

Below we run the locally cloned nanoGPT's `model.py`. We don't download GPT-2 weights or run full training — just use a very small configuration for a forward pass and loss.

Focus on two things when running:

1. Whether the `logits` shape is `[batch, seq, vocab_size]`.
2. Whether the randomly initialized model's loss is close to the random guess level.

In [ ]:
# Run nanoGPT's tiny configuration
from karpathy_models import NanoGPT, NanoGPTConfig

import torch
nano_config = NanoGPTConfig(
    block_size=16,
    vocab_size=64,
    n_layer=2,
    n_head=2,
    n_embd=32,
    dropout=0.0,
    bias=True,
)

torch.manual_seed(42)
nano_model = NanoGPT(nano_config)
nano_model.eval()

idx = torch.randint(0, nano_config.vocab_size, (2, 8))
targets = torch.randint(0, nano_config.vocab_size, (2, 8))

with torch.no_grad():
    nano_logits, nano_loss = nano_model(idx, targets)

print("=== nanoGPT tiny forward ===")
print(f"input shape:  {tuple(idx.shape)}")
print(f"logits shape: {tuple(nano_logits.shape)}")
print(f"loss:         {nano_loss.item():.4f}")
print(f"params:       {nano_model.get_num_params():,}")
print("Observation: logits are still [batch, seq, vocab_size].")

## 7. Structure and Behavior of nanoGPT

We only ran one forward pass earlier. In this section, we'll run a few small experiments with nanoGPT to observe how model structure affects parameter count, runtime, and initial loss:

1. How does parameter count change as the model gets deeper (more layers)?
2. How does forward pass time change as the sequence gets longer?
3. Is the randomly initialized model's loss close to the random guess level?

These experiments are not about proving the model "can generate" — it hasn't been trained yet. The focus is on observing how structure affects shapes, parameter counts, and computational cost.

In [ ]:
import matplotlib.pyplot as plt
# Experiment 1: How do nanoGPT parameters change as layers increase?
layer_counts = [1, 2, 4, 6]
nano_param_counts = []

for n_layer in layer_counts:
    cfg = NanoGPTConfig(
        block_size=16,
        vocab_size=64,
        n_layer=n_layer,
        n_head=2,
        n_embd=32,
        dropout=0.0,
        bias=True,
    )
    model = NanoGPT(cfg)
    nano_param_counts.append(model.get_num_params())

plt.figure(figsize=(7, 4))
plt.plot(layer_counts, nano_param_counts, marker="o")
plt.xlabel("Number of Transformer blocks")
plt.ylabel("Parameter count")
plt.title("nanoGPT tiny: parameters vs layers")
plt.grid(True, alpha=0.3)
plt.show()

for n_layer, params in zip(layer_counts, nano_param_counts):
    print(f"n_layer={n_layer}: params={params:,}")

print("Observation: more layers means Attention and MLP parameters in each Block increase approximately linearly.")

In [ ]:
import matplotlib.pyplot as plt
import time
# Experiment 2: How does forward time change as sequence length increases?
# Note: CPU timing has high variance, so each length is run multiple times for an average.
import torch
def benchmark_forward(model, seq_len, repeats=20):
    sample = torch.randint(0, nano_config.vocab_size, (2, seq_len))
    target = torch.randint(0, nano_config.vocab_size, (2, seq_len))
    with torch.no_grad():
        model(sample, target)  # warmup
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(repeats):
            model(sample, target)
    elapsed = time.perf_counter() - start
    return elapsed / repeats

seq_lengths = [4, 8, 12, 16]
forward_times = []
for seq_len_value in seq_lengths:
    forward_times.append(benchmark_forward(nano_model, seq_len_value))

plt.figure(figsize=(7, 4))
plt.plot(seq_lengths, forward_times, marker="o")
plt.xlabel("Sequence length")
plt.ylabel("Average forward time (seconds)")
plt.title("nanoGPT tiny: forward time vs sequence length")
plt.grid(True, alpha=0.3)
plt.show()

for seq_len_value, seconds in zip(seq_lengths, forward_times):
    print(f"seq_len={seq_len_value:2d}: avg_forward_time={seconds:.6f}s")

print("Observation: Self-Attention looks at pairwise token relationships, so longer sequences are typically more expensive.")

In [ ]:
import matplotlib.pyplot as plt
# Experiment 3: nanoGPT randomly initialized loss vs random guess level
# The model hasn't been trained, so loss should be close to log(vocab_size).
import torch
import math
torch.manual_seed(123)
compare_idx = torch.randint(0, 64, (4, 12))
compare_targets = torch.randint(0, 64, (4, 12))

with torch.no_grad():
    compare_logits, compare_loss = nano_model(compare_idx, compare_targets)

random_guess_loss = math.log(64)
losses = [compare_loss.item(), random_guess_loss]
names = ["nanoGPT", "random guess"]

plt.figure(figsize=(7, 4))
plt.plot(names, losses, marker="o")
plt.ylabel("Cross-Entropy loss")
plt.title("nanoGPT tiny: untrained loss vs random guess")
plt.grid(True, alpha=0.3)
plt.show()

print(f"nanoGPT loss:       {compare_loss.item():.4f}")
print(f"random guess loss:  {random_guess_loss:.4f}")
print("Observation: an untrained GPT's loss is close to random guess level. This is normal.")

## 8. Hand-written Version vs nanoGPT Comparison

Through the experiments above, let's put the correspondence between the hand-written version and nanoGPT side by side:

| Teaching Version | nanoGPT | Meaning |
|:---|:---|:---|
| `MiniGPT` | `GPT` | Complete model |
| `TransformerBlock` | `Block` | One decoder block |
| `MultiHeadAttention` | `CausalSelfAttention` | Causal self-attention |
| `FeedForward` | `MLP` | Per-token nonlinear processing |
| `token_emb` | `transformer.wte` | Token embedding |
| `pe` / position | `transformer.wpe` | Position embedding |
| `lm_head` | `lm_head` | Hidden state → vocab logits |

When reading real engineering code, it's recommended to follow along `GPT.forward`. As long as this main thread is clear, the engineering details in the training script (optimizer, checkpoint, learning rate schedule, etc.) won't cause confusion.

## 9. Special Tokens

MiniGPT can now output logits, but in practice the model also needs some "boundary markers" to indicate the structure of sequences.

For example: where a sequence starts, where it ends, and which positions are just for padding. In some scenarios, the model also needs to distinguish between "thinking process" and "final answer." These special markers are collectively called special tokens:

```text
<BOS>       begin of sequence, marks the start of a sequence
<EOS>       end of sequence, marks the end of a sequence; generation stops when this is encountered
<PAD>       padding, used to equalize lengths; marks that this position is not real content
🧠     thinking start, marks the beginning of a thinking section
🤔    thinking end, marks the end of a thinking section; the final answer follows
```

It's important to note that `🧠` does not make the model automatically become smarter. It's just a symbol — the model must learn to use it correctly, and the training data must repeatedly contain this format.

In [ ]:
# Demo: adding new special tokens to Mini-GPT's vocabulary
base_vocab = {
    "user": 0,
    "assistant": 1,
    "answer": 2,
    "357": 3,
    "289": 4,
    "103173": 5,
}

special_tokens = ["<BOS>", "<EOS>", "<PAD>", "🧠", "🤔"]

vocab = base_vocab.copy()
for token in special_tokens:
    if token not in vocab:
        vocab[token] = len(vocab)

print("New special token IDs:")
for token in special_tokens:
    print(f"  {token:8s} -> {vocab[token]}")

print()
print("Key observation: 🧠 and 🤔 now have independent IDs,")
print("so the model can treat them as boundary markers rather than ordinary text fragments.")

In [ ]:
# Demo: what a training sample with 🧠 looks like
sample_tokens = [
    "<BOS>",
    "user",
    "357",
    "289",
    "assistant",
    "🧠",
    "357",
    "289",
    "103173",
    "🤔",
    "answer",
    "103173",
    "<EOS>",
]

sample_ids = [vocab[token] for token in sample_tokens]

print("Training sample tokens:")
print(sample_tokens)
print()
print("Training sample IDs:")
print(sample_ids)
print()
print("Key observation: the model only learns when to start and stop thinking if the training data contains this format.")

After adding special tokens, the vocabulary grows. When the vocabulary grows, the Embedding table must also expand, because every token needs its own vector.

In [ ]:
# Small matrix demo: why Embedding must expand after adding new tokens
import torch
import torch.nn as nn
old_vocab_size = len(base_vocab)
new_vocab_size = len(vocab)
d_model = 8

torch.manual_seed(42)
old_embedding = nn.Embedding(old_vocab_size, d_model)
new_embedding = nn.Embedding(new_vocab_size, d_model)

# Copy old token vectors; new token vectors keep their random initialization,
# which will be updated during subsequent training
with torch.no_grad():
    new_embedding.weight[:old_vocab_size] = old_embedding.weight

print(f"Old vocabulary size: {old_vocab_size}")
print(f"New vocabulary size: {new_vocab_size}")
print(f"Embedding shape: {tuple(new_embedding.weight.shape)}")
print()
print("Key observation: after adding 5 special tokens, the Embedding has 5 more rows.")
print("These new rows must learn the actual purpose of 🧠 and 🤔 through subsequent training.")

## Summary

What we learned in this section:

- GPT = Embedding + Position + multi-layer Transformer Blocks + LayerNorm + output projection
- Intermediate hidden state shapes stay at `[batch, seq, d_model]`; only the final step changes to vocab_size
- Logits are scores for all tokens in the vocabulary at each position; higher scores mean the model prefers predicting that token
- The hand-written version and nanoGPT share the same structure; engineering code adds details like weight tying and learnable position embeddings
- Special tokens need independent IDs and corresponding Embedding table expansion

The next section moves into training: how to compute loss, and how the model goes from random initialization to generating meaningful text.

## Exercises

This exercise checks your understanding of the temperature parameter during generation.

> **About AI assistance**: You can ask AI to explain how temperature affects the probability distribution, but please write the line of code that scales the logits yourself.

**Exercise 1: Temperature-scaled logits**

Modern LLMs often use temperature to control randomness during generation. Lower temperature makes the distribution sharper (more deterministic); higher temperature makes it flatter (more random).

**Hint**: A common approach is `scaled_logits = logits / temperature`.

In [ ]:
# Exercise 1: temperature-scaled logits fill-in
import torch

logits = torch.tensor([1.0, 2.0, 3.0])
temperature = 0.5

# TODO: scale logits using temperature
scaled_logits = """Scale the logits here"""

assert not isinstance(scaled_logits, str), "Please replace the triple-quoted placeholder first"
assert torch.allclose(scaled_logits, torch.tensor([2.0, 4.0, 6.0])), scaled_logits
print("Exercise 1 passed: temperature < 1 makes logits spread wider, making the distribution sharper")

## References

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), 2017 — The original Transformer paper (Encoder-Decoder architecture); GPT is its Decoder-Only variant
- Harvard NLP, [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — A line-by-line implementation of the original Encoder-Decoder Transformer; useful as a "complete comparison" to understand the original architecture's Encoder/Decoder/Cross-Attention, then contrast with the Decoder-Only design in this notebook
- Karpathy, [nanoGPT](https://github.com/karpathy/nanoGPT) — The engineering implementation compared in this section
- Radford et al., [Language Models are Unsupervised Multitask Learners](https://d4mucfpksywv.cloudfront.net/better-language-models/language-models.pdf), 2019 — The GPT-2 paper